In [1]:
import numpy as np
import pandas as pd
import os
import ms_entropy as me


# Standard mapping for column names
standard_mapping = {
    "name": "Name",
    "ion_mode": "Ion Mode",
    "instrument": "Instrument",
    "instrument_type": "Instrument Type",
    "ionization": "Ionization",
    "collision_energy": "Collision Energy",
    "collision_gas": "Collision Gas",
    "sample_inlet": "Sample Inlet",
    "spectrum_type": "Spectrum Type",
    "precursor_type": "Precursor Type",
    "precursormz": "Precursor M/Z",
    "notes": "Notes",
    "inchikey": "InChIKey",
    "smiles": "SMILES",
    "synon": "Synonyms",
    "formula": "Formula",
    "mw": "Molecular Weight",
    "exactmass": "Exact Mass",
    "cas#": "CAS Number",
    "nist#": "NIST Number",
    "db#": "Database ID",
    "comments": "Comments",
    "unique_scan_number": "Unique Scan Number",
    "spectrum_id": "Spectrum ID",
    "num peaks": "Num Peaks"
}

def standardize_col(df):
    """
    Standardizes column names based on the provided mapping.

    Args:
        df (pd.DataFrame): The DataFrame whose column names need to be standardized.

    Returns:
        pd.DataFrame: DataFrame with standardized column names.
    """
    new_columns = []
    for col in df.columns:
        col_lower = col.lower().replace("reference_", "")
        standardized_col = standard_mapping.get(col_lower, col_lower)
        new_columns.append(standardized_col)
    df.columns = new_columns
    return df

def remove_zero_ions(msms):
    """
    Removes zero-intensity ions from an MS/MS spectrum.

    Args:
        msms (numpy.ndarray or float): MS/MS spectrum as a 2D NumPy array.

    Returns:
        numpy.ndarray: Filtered 2D array with non-zero intensities.
    """
    if isinstance(msms, float) or len(msms) == 0:
        return np.nan
    return msms[msms[:, 1] > 0]

def sort_spectrum(msms):
    """
    Sorts MS/MS peaks by m/z value.

    Args:
        msms (numpy.ndarray): 2D numpy array with [m/z, intensity].

    Returns:
        numpy.ndarray: Sorted 2D array.
    """
    if isinstance(msms, float) or len(msms) == 0:
        return np.nan
    return msms[np.argsort(msms[:, 0])]

def read_msp(file_path):
    """
    Parses an MSP file and extracts MS/MS spectra into a structured pandas DataFrame.

    Args:
        file_path (str): Path to the MSP file.

    Returns:
        pd.DataFrame: Parsed MS/MS spectra with metadata.
    """

    spectra = []
    spectrum = {}

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            if not line:
                continue  # Skip empty lines

            # Identify metadata fields
            if ":" in line:
                key, value = line.split(":", 1)
                key, value = key.strip().lower(), value.strip()

                if key == 'name':
                    # Save the previous spectrum before starting a new one
                    if spectrum:
                        spectra.append(spectrum)
                    spectrum = {'name': value, 'peaks': []}  # Start new spectrum entry
                
                else:
                    spectrum[key] = value  # Store metadata fields
            
            # Parse peak data (if line starts with a number, it's peak data)
            elif line[0].isdigit():
                try:
                    m_z, intensity = map(float, line.split()[:2])
                    spectrum['peaks'].append([m_z, intensity])
                except ValueError:
                    continue  # Ignore malformed lines

        # Save the last spectrum
        if spectrum:
            spectra.append(spectrum)

    # Convert spectra list to DataFrame
    df = pd.DataFrame(spectra)

    # Process peaks: Remove zero-intensity ions & sort by m/z
    df['peaks'] = [sort_spectrum(remove_zero_ions(np.array(peak))) for peak in df['peaks']]

    # Standardize column names
    df = standardize_col(df)

    # Convert numeric columns where applicable
    for column in df.columns:
        if column != 'peaks':  # Skip 'peaks' column
            try:
                df[column] = pd.to_numeric(df[column], errors='ignore')
            except:
                pass

    return df


In [2]:
df = pd.read_csv('/Users/ellayoung/Desktop/metabolo_confi_score/parsed_spectra.csv')

/var/folders/xd/3y34jslx1fsf1k1pd0bz8w3r0000gn/T/ipykernel_11449/446800534.py:1: DtypeWarning: Columns (11,28,29,30) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/Users/ellayoung/Desktop/metabolo_confi_score/parsed_spectra.csv')


In [3]:
df_clean = df.iloc[:, [0, 1, 10, 18, 13, 14, 16, 24]]

In [5]:
# only include entries with M+H or M-H adducts

df_clean_mh = df_clean[df_clean["Precursor Type"].isin(["[M+H]+", "[M-H]-"])]
df_unique = df_clean_mh.dropna(subset=['InChIKey'])
df_unique = df_unique.drop_duplicates(subset=['InChIKey'], keep='first')
# # Convert NaN or float SMILES to empty strings and ensure dtype is string
df_unique["SMILES"] = df_unique["SMILES"].fillna("").astype(str)

# Create a helper column with the first 14 characters of InChIKey
df_unique["inchi14"] = df_unique["InChIKey"].astype(str).str[:14]

In [6]:
def parse_peaks_string(peaks_str):
    """
    Parse a peaks string like:
    '[[136.0435   3.4   ]\n [214.9619  38.4   ]\n ...]'
    into a list of lists of floats.
    """
    # Remove the outer brackets and any leading/trailing whitespace
    cleaned = peaks_str.strip("[] \n")
    # Split into lines (each line corresponds to one peak row)
    rows = cleaned.split('\n')
    result = []
    for row in rows:
        # Remove any remaining brackets and extra whitespace
        row_clean = row.strip(" []")
        if row_clean:
            # Split on whitespace and convert each element to float
            values = row_clean.split()
            result.append([float(val) for val in values])
    return result

def safe_parse_peaks(peaks):
    """
    Safely parse peaks, returning an empty list for invalid cases.
    """
    if isinstance(peaks, str):
        if "..." in peaks or peaks.strip() == "":
            return []  # Return empty list for invalid entries
        try:
            return parse_peaks_string(peaks)
        except ValueError:
            return []  # Handle unexpected format issues
    return peaks  # Already parsed peaks



In [9]:
print(df_unique.head())

                                                  Name  \
0                            2-Guanidinopropionic acid   
5     (2-Hydroxymethylbicyclo[2.2.1]hept-2-yl)methanol   
68   2-Nitro-N-(propan-2-yl)-4-(trifluoromethyl)ani...   
115   4-Amino-1-methyl-5H-chromeno[3,4-c]pyridin-5-one   
125      1-Acetyl-2-methyl-1,2,3,4-tetrahydroquinoline   

                                                 peaks Precursor Type  \
0    [[ 53.0135   2.4   ]\n [ 55.018   55.2   ]\n [...         [M+H]+   
5    [[ 79.0542   9.5   ]\n [ 93.0699  34.9   ]\n [...         [M+H]+   
68   [[189.0272   4.6   ]\n [189.0635   9.    ]\n [...         [M+H]+   
115  [[115.0543   9.2   ]\n [128.0621   2.3   ]\n [...         [M+H]+   
125  [[106.0654   6.1   ]\n [120.0811   1.1   ]\n [...         [M+H]+   

     Exact Mass        InChIKey                                SMILES  \
0    131.069476  DVNFLGLGNLXITH                     CC(NC(=N)N)C(=O)O   
5    156.115029  RGXKKPWZFFCHNE                     OCC1(CO)CC2CCC

In [10]:
import pandas as pd
import itertools

# Assume df is your DataFrame
df_sorted = df.sort_values("Exact Mass").reset_index(drop=True)
isomer_pairs = []

# Iterate over all pairs (brute-force with early stopping)
for (idx1, row1), (idx2, row2) in itertools.combinations(df_sorted.iterrows(), 2):
    mz1, mz2 = row1["Exact Mass"], row2["Exact Mass"]

    # Skip if ppm difference is >10 ppm (sorted, so we can break early)
    ppm = abs(mz1 - mz2) / ((mz1 + mz2) / 2) * 1e6
    if ppm > 10:
        continue

    # Require different InChIKey cores
    if row1["inchi14"] != row2["inchi14"]:
        isomer_pairs.append((row1, row2))

print(f"Found {len(isomer_pairs)} isomer pairs with ±10 ppm and different InChI14.")


KeyError: 'inchi14'

In [ ]:
import itertools
import numpy as np
import random
from collections import defaultdict

# Function to compute null distribution using ±10 ppm pairs
def compute_null_distribution_ppm(df_unique, num_pairs=100, num_bootstrap=100, ppm_tolerance=10):
    """
    Compute a null distribution of similarity scores using ±10 ppm pairs,
    stratified by the entropy of the first spectrum in each pair.

    Parameters:
      df_unique (DataFrame): DataFrame containing spectral data with 'PrecursorMZ' and 'peaks' columns.
      num_pairs (int): Number of random pairs to sample per entropy group per bootstrap iteration.
      num_bootstrap (int): Number of bootstrap iterations to stabilize similarity estimates.
      ppm_tolerance (float): Precursor mass tolerance in ppm.

    Returns:
      dict: Null distributions of similarity scores grouped by spectral entropy.
    """
    entropy_groups = defaultdict(list)

    # Sort data by precursor mass for efficient ±10 ppm pairing
    df_sorted = df_unique.sort_values(by="Exact Mass").reset_index(drop=True)

    # List to hold ±10 ppm pairs
    ppm_pairs = []

    # Generate pairs within ±10 ppm window
    for i, row in df_sorted.iterrows():
        precursor_mz = row["Exact Mass"]
        mz_min = precursor_mz * (1 - ppm_tolerance / 1e6)
        mz_max = precursor_mz * (1 + ppm_tolerance / 1e6)

        # Find candidate spectra within the ±10 ppm window (excluding self-pairs)
        candidate_rows = df_sorted[(df_sorted["Exact Mass"] >= mz_min) & 
                                   (df_sorted["Exact Mass"] <= mz_max) &
                                   (df_sorted.index != i)]

        for _, candidate in candidate_rows.iterrows():
            ppm_pairs.append((row, candidate))

    print(f"Found {len(ppm_pairs)} pairs within ±{ppm_tolerance} ppm.")

    # Group by entropy
    for spec1, spec2 in ppm_pairs:
        peaks1_raw = spec1.get("peaks")
        peaks2_raw = spec2.get("peaks")

        try:
            parsed_peaks1 = safe_parse_peaks(peaks1_raw)
            parsed_peaks2 = safe_parse_peaks(peaks2_raw)
        except Exception as e:
            print(f"Error parsing peaks: {e}")
            continue

        spec1["peaks"] = parsed_peaks1
        spec2["peaks"] = parsed_peaks2

        if isinstance(parsed_peaks1, (list, np.ndarray)) and isinstance(parsed_peaks2, (list, np.ndarray)):
            entropy1 = me.calculate_spectral_entropy(np.array(parsed_peaks1, dtype=np.float32),
                                                     clean_spectrum=True,
                                                     min_ms2_difference_in_da=0.05)
            entropy_group = round(entropy1)
            entropy_groups[entropy_group].append((spec1, spec2))

    for entropy, pairs in entropy_groups.items():
        print(f"Entropy group {entropy} has {len(pairs)} pairs")

    # Create null distributions by bootstrapping from each entropy group
    null_distributions = defaultdict(list)

    for entropy, pairs in entropy_groups.items():
        if len(pairs) < 2:
            continue  # Skip groups with too few pairs

        for _ in range(num_bootstrap):
            if len(pairs) < num_pairs:
                selected_pairs = random.choices(pairs, k=num_pairs)
            else:
                selected_pairs = random.sample(pairs, num_pairs)

            scores = [
                me.calculate_entropy_similarity(spec1["peaks"], spec2["peaks"], ms2_tolerance_in_da=10)
                for spec1, spec2 in selected_pairs
            ]
            null_distributions[entropy].extend(scores)

    return null_distributions
